# Structured Extraction from Blatten News with an LLM

## What this notebook does

This notebook extracts structured information from news articles about the Blatten (Lötschental) rockfall/debris-flow event. It uses an LLM (via LangChain + OpenAI) with a **Pydantic schema** to pull out, for each article:

- **event** — the type of natural event (e.g. debris flow, rockfall, avalanche, landslide, flood)
- **sentiment** / **tone** — positive/negative/neutral, aggressive/peaceful/neutral
- **location** and **name** — places and people mentioned
- **strategy** — response measures (road closure, relocation, monitoring, early warning, ...)
- **analysis** — stated causes (e.g. permafrost thawing, glacial retreat, climate change)
- **stories** — short synthesis of personal accounts related to the event
- **frames** — the interpretive frame(s) the article uses
- **topic** — a short English keyword summary of the article's topic

All output is normalized to English regardless of the source language (German, English, French, Italian).

## Workflow (run top to bottom)

1. **Install & update** — installs `langchain`, `langchain-openai`, `beautifulsoup4`, `dask`, `faiss-cpu`.
2. **Set your OpenAI API key** — you'll be prompted via `getpass` if `OPENAI_API_KEY` isn't already set in the environment.
3. **Prompt and tagging model** — defines the extraction prompt and the `Data` / `General` / `Topic` Pydantic schema, then builds `tagging_chain` (prompt → LLM with structured output). A sample German passage is run through it as a sanity check.
4. **Load files and clean up** — reads the raw news CSV, strips HTML, deduplicates exact-duplicate content, generates multilingual sentence embeddings, and uses FAISS to also drop *near*-duplicate articles.
5. **Parallel tagging (time-consuming!)** — runs `tagging_chain` over every remaining article using a thread pool, then flattens the results into new columns on the dataframe.
6. **Save output** — writes the tagged dataframe to a CSV (no translation applied at this stage).

## How to use it

- **Before running:** make sure the input file path in the "Load files and clean up" section (`cleaned_news_4LLM_2.csv`) matches where your data actually lives, and update the output paths similarly if needed.
- **API key:** have your OpenAI API key ready — you'll be asked for it once per session (it's cached for the rest of the run).
- **Model/cost note:** the tagging model is set in the "Prompt and tagging models" cell (currently `gpt-5-nano`). Adjust there if you want a different model.
- **Testing vs. full run:** the sampling cell (`df_select = unique_df.sample(frac=1, ...)`) currently uses 100% of the deduplicated data. Lower `frac` while testing, since the parallel tagging step calls the LLM once per article and can take a long time / incur cost on the full dataset.
- **Re-running safely:** the cleaning, embedding, and deduplication steps each check whether their output file already exists and skip recomputation if so — delete the corresponding file if you want to force a rebuild.
- **Output:** the final tagged CSV is saved with one row per article and the extracted fields as columns, ready for downstream analysis (e.g. by the summarization/topic-analysis notebooks in this folder).

*This notebook was originally developed by Veruska and Saeid and adapted by Veruska for the Blatten news analysis; it focuses on analysing Blatten news extracted via UZH LiRI.*

## Install and update


In [ ]:
from langchain_openai import ChatOpenAI
import os
import getpass
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, validator
from bs4 import BeautifulSoup
import dask.dataframe as dd
import faiss
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [ ]:
ChatOpenAI.api_key = os.getenv('OPENAI_API_KEY')

## Prompt and tagging models

In [ ]:
# Define prompt template for extracting structured information using the 'Data' schema.
tagging_prompt = ChatPromptTemplate.from_template(
    """
Extract the desired information from the following dataset.

Only extract the information mentioned in the 'Data' function.

The information could be in the following languages: German, English, French or Italian.

Please make sure you always provide your output in English and make sure you translate accurately.

Do not make up content. Rigorously extract only the required information.


Passage:
{input}
"""
)

In [ ]:
# This schema is to analyse multilingual news with an open vocabulary approach.
# The schema is designed to be used with the LangChain framework.

class General(BaseModel):
    event: Optional[str] = Field(default= None, description="""any natural event due to climate, weather or geological event if any. Possible event could be:
                                                              'debris flows', 'rockfall', 'avalanche', 'landslides', 'extreme precipitation', 'flood'
                                                               If more than one please separate with semicolumn.""")
    sentiment: str = Field(
        description='the sentiment of the news.',
        enum=["positive", "negative", "neutral"],
    )
    tone: str = Field(
        description='the tone of the news.', enum=["aggressive", "peaceful", "neutral"]
    )
    location: Optional[str] = Field(default = None, description="The location or locations mentioned in the news if any. "
    )
    name: Optional[str] = Field(default = None, description="""Name of persons mentioned in the text. If more than one separate them by semicolumn.
                                Name are only identified with person's names such as Name and Surname, Surname and Name,
                                or the name only with intial and full stop and then surname, for example A. Smith. Or in could be M. and then the Surname.
                                Some people have two surnames. Please make sure you distinsuish people's name from other things in particular when the language
                                is German. Do not report other names that are not person's names.
                                Name of institutions or desciptions of people without proper name should not be classified as name"""
    )
    strategy: Optional[str] = Field(default = None, description="""
                      Some news mention measures, plans taken or that will be taken to address the situation described in the event.
                      Please report any response, plans, measures, strategies both short and long term. Measures could be:
                      'road closure', 'relocation of people', 'monitoring', 'early warning'. If more than one is mentioned, please
                      provide them separated  by semicolumn.
                      Do not make up text if you don't find any relevant information.""")
    analysis: Optional[str] = Field(default = None, description="""
                      Some news mention environmental analysis of the event. Please provide any relevant information that discusses
                      the causes of the event, for example:
                      "long-term geological instability", "thawing permafrost", "glacial retreat", "climate change" and other possible cause.
                      Do not make up text if you don't find any relevant information.""")
    stories: Optional[str] = Field(default = None, description="""
                      Provide very short synthesis of stories and factual an described by people in relation to the event.
                      This could be people directly or indirectly involved in the events.""")
    frames: Optional[str] = Field(default = None, description="""Provide a short description of the frames emerging in the news article.
                      A frame is defined as follows
                      "A recurring, repetitive (across and within articles),
                      overt or implicit claim that presents and promotes a specific
                      interpretation or viewpoint on an ongoing (and frequently dynamic) news topic"

                      If more than one frame emerge in the news article, provide all of them separated by semicolumn.
                      It's important not to repeat the same frame more than once for the same news article.

                      """)

class Topic(BaseModel):
    """Information about the topic or topics of the news."""

    # Doc-string for the entity Impact.
    # This doc-string is sent to the LLM as the description of the schema Impact,
    # and it can help to improve extraction results.

    # Note that:
    # 1. Each field is an `optional` -- this allows the model to decline to extract it!
    # 2. Each field has a `description` -- this description is used by the LLM.
    # Having a good description can help improve extraction results.
    topic: Optional[str] = Field(default=None, description="""
    Topic or topics of the content. Please summarise very briefly with some key words. The summary should always be in "english" even if the text is not in english.
    """)

class Data(BaseModel):
    """Extracted data about general information and topics."""

    # Creates a model so that we can extract multiple entities.
    general: List[General]
    topic: List[Topic]

In [ ]:
# Initialize the LLM with the structured output defined by the 'Data' schema and chain it with the prompt.
llm = ChatOpenAI(temperature=0, model="gpt-5-nano-2025-08-07").with_structured_output(Data)
tagging_chain = tagging_prompt | llm  #make sure you refer to the correct prompt

In [ ]:
# Test the tagging chain by invoking it with a sample input and print the extracted results.
# Be careful to choose the right schema corresponding to the chosen prompt.
inp2 = """

Ein in den vergangenen Tagen abgegangener Murgang im Lötschental zwang die Bewohnerinnen
und Bewohner von Blatten VS, ihr Zuhause zu verlassen. Bewohnerinnen und Bewohner des
Dorfes Blatten VS im Lötschental haben am Samstagabend ihre Wohnungen verlassen müssen.
Grund dafür war ein in den vergangenen Tagen abgegangener Murgang.
Die Evakuation sei von den kantonalen Behörden angeordnet worden,
wie Gemeinderat Reto Kalbermatten gegenüber der Nachrichtenagentur Keystone-SDA mitteilte.
Weitere Informationen sollen am Sonntag um 14.00 Uhr bekannt gegeben werden,
wie Kalbermatten erklärte. Ein Felssturz im Gebiet des «Kleinen Nesthorns»,
der auch einen Teil des Birchgletschers mitriss, habe vor einigen Tagen einen Murgang ausgelöst.
Dieser sei von der Grösse einer kleinen Lawine gewesen und rund
500 Meter oberhalb des Flusses Lonza ausserhalb des Dorfes zum Stillstand gekommen.

"""
run = tagging_chain.invoke({"input": inp2})
print(run)

In [ ]:
print(run.general[0].event)
print(run.general[0].sentiment)
print(run.general[0].tone)
print(run.general[0].location)
print(run.general[0].name)
print(run.general[0].strategy)
print(run.general[0].stories)
print(run.general[0].analysis)
print(run.topic[0].topic)
print(run.general[0].frames)

## Load files and clean up

In [ ]:
# Define the function to remove HTML tags
def remove_html_tags(text):
    if not isinstance(text, str):
        return text
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

file_path = '/Users/muccione/Documents/ManuscriptOngoing/Blatten/data/news_predictions_all_relevant_20251201.csv'
output_file_path = '/Users/muccione/Documents/ManuscriptOngoing/Blatten/data/cleaned_no_duplicates.parquet'


# Check if the cleaned file already exists
if os.path.exists(output_file_path):
    print(f"Cleaned file already exists at {output_file_path}. Skipping processing...")
    ddf = dd.read_parquet(output_file_path)  # Load the existing cleaned dataset
else:
    print("Cleaned file not found. Processing the dataset...")

    # Define column data types
    dtypes = {
        'id': 'int',
        'pubtime' : 'object',
        'medium_code': 'object',
        'medium_name': 'object',
        'rubric': 'object',
        'regional': 'object',
        'doctype': 'object',
        'doctype_description': 'object',
        'language':  'object',
        'char_count': 'int',
        'dateline': 'object',
        'head': 'object',
        'subhead': 'object',
        'article_link': 'object',
        'content_id': 'object',
        'content': 'object',
        'clean_content': 'object',


     #  'Research Data DOI': 'object',
     #  'Research Data Repository': 'object',
    }
    # Load the dataset
    ddf = dd.read_csv(file_path, delimiter=',',encoding='utf-8',dtype=dtypes)
    print(ddf.columns)
    print(ddf.head(10))
    
    # Step 1: Remove HTML tags
    ddf['content'] = ddf['clean_content'].map_partitions(lambda df: df.apply(remove_html_tags))

    # Step 2: Remove duplicates
    ddf = ddf.drop_duplicates(subset=['content'])

    # Save the cleaned and deduplicated dataset as a Parquet file
    ddf.to_parquet(output_file_path, engine='pyarrow')
    print(f"Cleaned and deduplicated dataset saved to {output_file_path}")

In [ ]:
# Define the path to save/load embeddings

embeddings_path = '/content/drive/MyDrive/deep_learning_7_SV/Blatten_news/embeddings.npy'

# Check if embeddings already exist
if os.path.exists(embeddings_path):
    print("Embeddings file found. Loading embeddings...")
    embeddings_np = np.load(embeddings_path)
else:
    print("Embeddings file not found. Generating embeddings...")

    # Load the cleaned and deduplicated dataset
    cleaned_file_path = '/content/drive/MyDrive/deep_learning_7_SV/Blatten_news/cleaned_no_duplicates.parquet'
    ddf = dd.read_parquet(cleaned_file_path)
    df = ddf.compute()  # Convert to pandas DataFrame

    # Define a function to encode text in batches
    def encode_in_batches(texts, model, batch_size=64):
        """Encodes text in smaller batches to manage memory usage efficiently."""
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            embeddings.append(model.encode(batch, convert_to_tensor=True))
        return torch.cat(embeddings)

    # Load the model and encode the `content` column
    model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
    data = df['content'].tolist()

    # Encode content in batches
    embeddings = encode_in_batches(data, model)

    # Convert embeddings to NumPy and save to disk
    embeddings_np = embeddings.cpu().numpy()
    np.save(embeddings_path, embeddings_np)
    print(f"Embeddings saved to {embeddings_path}")

In [ ]:
# File path for the output dataset
output_parquet_path = '/content/drive/MyDrive/deep_learning_7_SV/Blatten_news/unique_dataset.parquet'

# Check if the unique dataset file already exists
if os.path.exists(output_parquet_path):
    print(f"Unique dataset already exists at {output_parquet_path}. Skipping FAISS processing...")
    unique_df = pd.read_parquet(output_parquet_path)  # Load the deduplicated dataset
else:
    print("Unique dataset not found. Running FAISS to deduplicate...")

    # Perform nearest neighbor search using FAISS
    # - FAISS helps identify near-duplicate rows using embeddings.
    # - L2 distance is used to measure similarity between embeddings.
    index = faiss.IndexFlatL2(embeddings_np.shape[1])  # Create FAISS index
    index.add(embeddings_np)  # Add embeddings to the FAISS index

    # Query for similar items
    # - Retrieves the top 2 nearest neighbors for each embedding.
    D, I = index.search(embeddings_np, 2)

    # Keep only the first occurrences based on a similarity threshold
    keep_mask = [True] * len(embeddings_np)  # Initialize mask to keep all rows
    for idx, (distances, indices) in enumerate(zip(D, I)):
        if distances[1] < 0.10:  # Adjust this threshold as needed
            keep_mask[indices[1]] = False  # Mark near-duplicates for removal

    # Calculate metrics for dropped rows
    total_rows = len(embeddings_np)
    kept_rows = sum(keep_mask)
    dropped_rows = total_rows - kept_rows
    dropped_percentage = (dropped_rows / total_rows) * 100

    # Log the metrics
    print(f"Total rows: {total_rows}")
    print(f"Rows kept: {kept_rows}")
    print(f"Rows dropped: {dropped_rows}")
    print(f"Percentage of rows dropped: {dropped_percentage:.2f}%")

    # Create a DataFrame with unique rows
    unique_df = df[keep_mask]

    # Save the unique dataset to Parquet
    unique_df.to_parquet(output_parquet_path, engine='pyarrow', index=False)
    print(f"Unique dataset saved to {output_parquet_path}")


In [ ]:
# Sampling 10% of the dataset
# - Sampling reduces the size of the dataset for quicker testing or tagging.
# - The `frac=0.1` parameter selects 10% of the rows from the dataset.
# - `random_state=42` ensures reproducibility of the random sampling process.
df_select = unique_df.sample(frac=1, random_state=42) #sample less if you want to do for testing
#df_select = unique_df

## Parallel tagging - Time consuming!!

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# Define a safe tagging function
# - Safely invokes the tagging chain for individual rows.
# - Handles errors gracefully to ensure the process continues even if some rows fail.
def safe_tagging(content):
    try:
        return tagging_chain.invoke({"input": content})
    except Exception as e:
        return {"error": str(e)}  # Return error details for debugging

# Function to tag the dataset in parallel using ThreadPoolExecutor
# - Tags all rows in the dataset using multiple threads for faster execution.
# - `max_workers` controls the number of parallel threads.
# - Tracks progress with `tqdm`.
def parallel_tagging(data, max_workers=8):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(safe_tagging, data), total=len(data), desc="Tagging"))
    return results

# Perform tagging for the entire dataset
results = parallel_tagging(df_select['content'], max_workers=8)

# Log the number of successfully tagged rows
print(f"Tagging completed for {len(results)} rows.")

In [ ]:
print(run.general[0].event)
print(run.general[0].sentiment)
print(run.general[0].tone)
print(run.general[0].location)
print(run.general[0].name)
print(run.general[0].strategy)
print(run.general[0].stories)
print(run.general[0].analysis)
print(run.topic[0].topic)
print(run.general[0].frames)

In [ ]:
# Extract results safely by checking attributes
# - Accesses attributes directly from the `Data` objects in the results list.
# - Uses `hasattr` and `getattr` to handle missing attributes gracefully.
# - This ensures robustness in case tagging fails for some rows.

df_select['event'] = [
    getattr(result.general[0], 'event', None) if hasattr(result, 'general') and result.general else None
    for result in results
    ]
df_select['sentiment'] = [
    getattr(result.general[0], 'sentiment', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['tone'] = [
    getattr(result.general[0], 'tone', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['location'] = [
    getattr(result.general[0], 'location', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['name'] = [
    getattr(result.general[0], 'name', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['strategy'] = [
    getattr(result.general[0], 'strategy', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['stories'] = [
    getattr(result.general[0], 'stories', None) if hasattr(result, 'general') and result.general else None
    for result in results
]
df_select['analysis'] = [
    getattr(result.general[0], 'analysis', None) if hasattr(result, 'general') and result.general else None
    for result in results
]

df_select['frames'] = [
    getattr(result.general[0], 'frames', None) if hasattr(result, 'general') and result.general else None
    for result in results
]

df_select['topic'] = [
    getattr(result.topic[0], 'topic', None) if hasattr(result, 'topic') and result.topic else None
    for result in results
]

In [ ]:
#At this stage you could save the file (no translation yet)
final_output_path = '../../Blatten/data/news_predictions_all_relevant_tagged_20251201.csv'
df_select.to_csv(final_output_path, index=False, encoding='utf-8')

# Log the save operation
print(f"Processed dataset saved to {final_output_path}")

In [ ]:
#Prepare a sample for evaluation
df_eval = df_select.sample(215)
df_eval.to_csv("../../Blatten/data/news_predictions_all_relevant_tagged_EVAL.csv", index=False)